# 1.5 · Backtesting

**Objetivo:** probar una estrategia de cruce de medias móviles sin look-ahead bias.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # repo root, so `common` is importable
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import *
set_plot_style()

In [ ]:
px = download_prices("SPY", start="2005-01-01").iloc[:, 0]
ret = px.pct_change().dropna()

## Ejercicio — Señal y posición
Señal: 1 si SMA(50) > SMA(200), si no 0. **Desplaza la posición 1 día** para evitar look-ahead.

In [ ]:
# TODO: position, strat_ret


In [ ]:
sma_fast, sma_slow = px.rolling(50).mean(), px.rolling(200).mean()
signal = (sma_fast > sma_slow).astype(int)
position = signal.shift(1).reindex(ret.index).fillna(0)
cost = 0.0005 * position.diff().abs().fillna(0)   # 5 bps por cambio de posición
strat_ret = position * ret - cost

## Ejercicio — Métricas y comparación con buy & hold

In [ ]:
eq = pd.DataFrame({"Estrategia": (1 + strat_ret).cumprod(), "Buy & Hold": (1 + ret).cumprod()})
eq.plot(logy=True, title="Equity curve"); plt.show()
summary = pd.DataFrame({
    "Retorno anual": [annualized_return(strat_ret), annualized_return(ret)],
    "Vol anual": [annualized_vol(strat_ret), annualized_vol(ret)],
    "Sharpe": [sharpe_ratio(strat_ret), sharpe_ratio(ret)],
    "Max DD": [max_drawdown(eq["Estrategia"]), max_drawdown(eq["Buy & Hold"])],
}, index=["Estrategia", "Buy & Hold"])
summary.round(3)